# Matrix Multiplication Runtime Data Movement Visualization

## Overview

The purpose of this notebook is to visually describe how the A, B, and C matrices are tiled and transformed in the matrix multiply whole array design.

The design tiles each matrix once, with `tile_matrices()`. A tiling is itself a `TensorAccessPattern` (from taplib, the Tensor Access Pattern Library in IRON) whose leading dimensions index the tiles, so indexing or slicing those dimensions picks out the tiles one DMA transfer moves. `step_transfers()` lists those slices for one row block of C, and the runtime sequence fills or drains each of them. This notebook imports both functions from `whole_array.py` and plots the patterns with `TensorAccessPattern.visualize()` and `animate()`. Only access patterns are built, so it needs neither a compiler nor an NPU.

## Tile the matrices

In [ ]:
from whole_array import step_transfers, tile_matrices

from IPython.display import HTML

M = 640
K = 896
N = 768
m = 16
k = 32
n = 48
n_aie_cols = 4

tilings = tile_matrices(
    M, K, N, m, k, n, n_aie_cols, b_col_maj=False, c_col_maj=False
)
for name, tiling in zip("ABC", tilings):
    print(name, tiling)

`tilings.A` has sizes `[M // m, N // n // n_aie_cols, K // k, m, k]`: `tilings.A[i]` is the i-th row of A tiles, walked once for every column of output tiles a shim column computes. `tilings.B` has sizes `[N // n, K // k, k, n]`, one column of B tiles per index of its first dimension. `tilings.C` has sizes `[M // (4 * m), N // n, 4 * m, n]`: `tilings.C[i, j]` is the C tile the four cores of column `j` produce for row block `i`.

## How to Read These Graphs

Note: It can take some time (order of seconds) to generate each graph.

The ***Access Order*** graph is a heatmap showing the order that data is conceptually pushed or pulled off the wires in DMA operations. The graph goes from cold (black) to hot (white).

The ***Access Count*** graph shows how many times each element is accessed by the pattern. This is also a heatmap; when every element is accessed the same number of times (the max value is displayed in the graph title) it is a single color.

## 1. The whole tilings

In [ ]:
tilings.A.visualize(title="A", plot_access_count=True)

Every element of A is accessed `N // n // n_aie_cols` times: the `repeat()` in its tiling.

In [ ]:
tilings.B.visualize(title="B", plot_access_count=True)

In [ ]:
tilings.C.visualize(title="C", plot_access_count=True)

## 2. One row block's transfers

The runtime sequence issues `step_transfers(tilings, row, n_aie_cols)` for each row block `row`. Shim column `col` drains every `n_aie_cols`-th C tile of the row block, fills the row of A tiles its core row multiplies, and fills every `n_aie_cols`-th column of B. These are shim column 0's transfers for row block 0:

In [ ]:
for tensor, col, tap in step_transfers(tilings, 0, n_aie_cols):
    if col == 0:
        print(tensor, tap)
        tap.visualize(title=f"{tensor}, shim column {col}")

# Animated tilings

`animate(frame_dims=1)` draws one frame per index of a tiling's first dimension: one row of A tiles, one column of B tiles, or one row block of C.

Note: It can take some time (order of minutes) to generate each animation.

## 1. A, one row of tiles per frame

In [ ]:
a_anim = tilings.A.animate(frame_dims=1, title="A")
HTML(a_anim.to_jshtml())

## 2. B, one column of tiles per frame

In [ ]:
b_anim = tilings.B.animate(frame_dims=1, title="B")
HTML(b_anim.to_jshtml())

## 3. C, one row block per frame

In [ ]:
c_anim = tilings.C.animate(frame_dims=1, title="C")
HTML(c_anim.to_jshtml())